# Análisis integrado FGJ + pobreza multidimensional

Este notebook está pensado para:

1. Leer de forma robusta los CSV grandes.
2. Limpiar y diagnosticar calidad de datos.
3. Filtrar correctamente CDMX en la base de pobreza.
4. Construir paneles por alcaldía-año.
5. Delimitar un bloque principal de delitos **patrimoniales/robos territoriales** y un bloque secundario de **fraude**.
6. Generar métricas reportables, asociaciones y un componente predictivo exploratorio.

## Enfoque metodológico sugerido
- **Objetivo principal**: análisis explicativo territorial.
- **Objetivo secundario**: predicción exploratoria del nivel de robos territoriales por alcaldía.
- **Ventana principal**: años comunes detectados automáticamente entre pobreza y FGJ.
- **Ventana secundaria**: años posteriores al último año común, usando pobreza estructural promedio del periodo común.

In [1]:
from pathlib import Path
import json
import re
import unicodedata
from collections import Counter
from itertools import cycle

import numpy as np
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, classification_report, confusion_matrix, accuracy_score, balanced_accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 120)

## Configuración de rutas

Este bloque detecta si estás en la raíz del proyecto o dentro de `scripts/`.
Ajusta los nombres de archivo si tus CSV tienen otro nombre.

In [2]:
if (Path.cwd() / "datasets").exists():
    BASE_DIR = Path.cwd()
elif (Path.cwd().parent / "datasets").exists():
    BASE_DIR = Path.cwd().parent
else:
    raise FileNotFoundError("No se encontró la carpeta 'datasets' desde el directorio actual ni desde su padre.")

DATA_DIR = BASE_DIR / "datasets"
OUT_DIR = BASE_DIR / "Documentation" / "eda_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

FGJ_FILE = DATA_DIR / "carpetasFGJ_acumulado_2025_01.csv"
POBREZA_FILE = DATA_DIR / "enigh_16_20.csv"

FGJ_CHUNKSIZE = 200_000

print("BASE_DIR:", BASE_DIR)
print("FGJ_FILE:", FGJ_FILE, "| Existe:", FGJ_FILE.exists())
print("POBREZA_FILE:", POBREZA_FILE, "| Existe:", POBREZA_FILE.exists())
print("OUT_DIR:", OUT_DIR)

BASE_DIR: d:\DMProject
FGJ_FILE: d:\DMProject\datasets\carpetasFGJ_acumulado_2025_01.csv | Existe: True
POBREZA_FILE: d:\DMProject\datasets\enigh_16_20.csv | Existe: True
OUT_DIR: d:\DMProject\Documentation\eda_outputs


## Utilidades

In [3]:
def normalize_text(value):
    if pd.isna(value):
        return None
    text = str(value).strip().upper()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = re.sub(r"[^A-Z0-9]+", " ", text)
    text = " ".join(text.split())
    return text or None

ALCALDIA_ALIASES = {
    "GUSTAVO A MADERO": "GUSTAVO A MADERO",
    "MAGDALENA CONTRERAS": "LA MAGDALENA CONTRERAS",
    "LA MAGDALENA CONTRERAS": "LA MAGDALENA CONTRERAS",
    "ALVARO OBREGON": "ALVARO OBREGON",
    "MIGUEL HIDALGO": "MIGUEL HIDALGO",
    "VENUSTIANO CARRANZA": "VENUSTIANO CARRANZA",
    "BENITO JUAREZ": "BENITO JUAREZ",
    "CUAJIMALPA DE MORELOS": "CUAJIMALPA DE MORELOS",
    "CUAUHTEMOC": "CUAUHTEMOC",
    "COYOACAN": "COYOACAN",
    "AZCAPOTZALCO": "AZCAPOTZALCO",
    "IZTACALCO": "IZTACALCO",
    "IZTAPALAPA": "IZTAPALAPA",
    "MILPA ALTA": "MILPA ALTA",
    "TLAHUAC": "TLAHUAC",
    "TLALPAN": "TLALPAN",
    "XOCHIMILCO": "XOCHIMILCO",
}

MAP_MUNICIPIO_CDMX = {
    2: "AZCAPOTZALCO",
    3: "COYOACAN",
    4: "CUAJIMALPA DE MORELOS",
    5: "GUSTAVO A MADERO",
    6: "IZTACALCO",
    7: "IZTAPALAPA",
    8: "LA MAGDALENA CONTRERAS",
    9: "MILPA ALTA",
    10: "ALVARO OBREGON",
    11: "TLAHUAC",
    12: "TLALPAN",
    13: "XOCHIMILCO",
    14: "BENITO JUAREZ",
    15: "CUAUHTEMOC",
    16: "MIGUEL HIDALGO",
    17: "VENUSTIANO CARRANZA",
}

def normalize_alcaldia(value):
    txt = normalize_text(value)
    if txt is None:
        return None
    return ALCALDIA_ALIASES.get(txt, txt)

def norm_col(col):
    col = normalize_text(col)
    return col if col is not None else ""

def resolve_column(columns, aliases):
    cols_map = {norm_col(c): c for c in columns}
    for alias in aliases:
        alias_norm = norm_col(alias)
        if alias_norm in cols_map:
            return cols_map[alias_norm]
    return None

def to_jsonable(obj):
    if isinstance(obj, dict):
        return {str(k): to_jsonable(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [to_jsonable(v) for v in obj]
    elif isinstance(obj, tuple):
        return [to_jsonable(v) for v in obj]
    elif isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    elif isinstance(obj, np.bool_):
        return bool(obj)
    elif isinstance(obj, pd.Timestamp):
        return obj.isoformat()
    elif isinstance(obj, pd.Series):
        return to_jsonable(obj.to_dict())
    elif isinstance(obj, pd.DataFrame):
        return to_jsonable(obj.to_dict(orient="records"))
    elif pd.isna(obj):
        return None
    else:
        return obj

def save_csv(df, name):
    path = OUT_DIR / name
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print("[OK] Guardado CSV:", path)

def save_json(obj, name):
    path = OUT_DIR / name
    with open(path, "w", encoding="utf-8") as f:
        json.dump(to_jsonable(obj), f, ensure_ascii=False, indent=2)
    print("[OK] Guardado JSON:", path)

def save_txt(text, name):
    path = OUT_DIR / name
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)
    print("[OK] Guardado TXT:", path)

def weighted_mean(series, weights):
    s = pd.to_numeric(series, errors="coerce")
    w = pd.to_numeric(weights, errors="coerce")
    mask = s.notna() & w.notna()
    if mask.sum() == 0:
        return np.nan
    if w[mask].sum() == 0:
        return np.nan
    return float(np.average(s[mask], weights=w[mask]))

def weighted_mode(series, weights):
    s = series.copy()
    w = pd.to_numeric(weights, errors="coerce")
    mask = s.notna() & w.notna()
    if mask.sum() == 0:
        return np.nan
    tmp = pd.DataFrame({"valor": s[mask], "peso": w[mask]}).groupby("valor", dropna=False)["peso"].sum().sort_values(ascending=False)
    return tmp.index[0] if len(tmp) else np.nan

ENCODINGS_TO_TRY = ["utf-8", "utf-8-sig", "cp1252", "latin1"]

def read_csv_flexible(path, nrows=None, usecols=None, chunksize=None, low_memory=False):
    last_error = None
    for enc in ENCODINGS_TO_TRY:
        try:
            obj = pd.read_csv(
                path,
                encoding=enc,
                nrows=nrows,
                usecols=usecols,
                chunksize=chunksize,
                low_memory=low_memory
            )
            print(f"[OK] Archivo leído con encoding: {enc}")
            return obj, enc
        except Exception as e:
            last_error = e
    raise last_error

## Paso 1. Encabezados y resolución de columnas clave

In [4]:
fgj_header, fgj_enc_header = read_csv_flexible(FGJ_FILE, nrows=0)
pob_header, pob_enc_header = read_csv_flexible(POBREZA_FILE, nrows=0)

fgj_cols = fgj_header.columns.tolist()
pob_cols = pob_header.columns.tolist()

# FGJ
FGJ_COLS = {
    "anio_hecho": resolve_column(fgj_cols, ["anio_hecho", "año_hecho"]),
    "anio_inicio": resolve_column(fgj_cols, ["anio_inicio", "año_inicio"]),
    "fecha_hecho": resolve_column(fgj_cols, ["fecha_hecho"]),
    "fecha_inicio": resolve_column(fgj_cols, ["fecha_inicio"]),
    "delito": resolve_column(fgj_cols, ["delito"]),
    "categoria": resolve_column(fgj_cols, ["categoria_de_competencia", "categoría_de_competencia"]),
    "alcaldia_hecho": resolve_column(fgj_cols, ["alcaldia_hech", "alcaldia_hecho"]),
    "alcaldia_cat": resolve_column(fgj_cols, ["alcaldia_cata", "alcaldia_cat"]),
    "municipio": resolve_column(fgj_cols, ["municipio_he", "municipio_h", "municipio"]),
    "latitud": resolve_column(fgj_cols, ["latitud"]),
    "longitud": resolve_column(fgj_cols, ["longitud"]),
}

# Pobreza
POB_COLS = {
    "anio": resolve_column(pob_cols, ["año", "anio"]),
    "entidad": resolve_column(pob_cols, ["entidad", "cve_ent", "ent"]),
    "municipio": resolve_column(pob_cols, ["municipio", "cve_mun", "mun"]),
    "cvegeo": resolve_column(pob_cols, ["cvegeo", "cve_mun_ent", "cveg", "cvegj"]),
    "factor": resolve_column(pob_cols, ["factor", "fac"]),
    "mmip": resolve_column(pob_cols, ["mmip"]),
    "rei": resolve_column(pob_cols, ["rei"]),
    "nbi": resolve_column(pob_cols, ["nbi"]),
    "CBDj": resolve_column(pob_cols, ["CBDj", "cbdj"]),
    "CSj": resolve_column(pob_cols, ["CSj", "csj"]),
    "CTEU": resolve_column(pob_cols, ["CTEU", "cteu"]),
    "CENJ": resolve_column(pob_cols, ["CENJ", "cenj"]),
    "ett": resolve_column(pob_cols, ["ett"]),
    "CASi": resolve_column(pob_cols, ["CASi", "casi"]),
    "CASSi": resolve_column(pob_cols, ["CASSi", "cassi"]),
    "cyt": resolve_column(pob_cols, ["cyt"]),
    "ccevj": resolve_column(pob_cols, ["ccevj"]),
    "E_mmip": resolve_column(pob_cols, ["E_mmip", "e_mmip"]),
    "E_rei": resolve_column(pob_cols, ["E_rei", "e_rei"]),
    "E_nbi": resolve_column(pob_cols, ["E_nbi", "e_nbi"]),
}

save_json({"FGJ_COLS": FGJ_COLS, "POB_COLS": POB_COLS}, "columnas_resueltas.json")
FGJ_COLS, POB_COLS

[OK] Archivo leído con encoding: utf-8
[OK] Archivo leído con encoding: cp1252
[OK] Guardado JSON: d:\DMProject\Documentation\eda_outputs\columnas_resueltas.json


({'anio_hecho': 'anio_hecho',
  'anio_inicio': 'anio_inicio',
  'fecha_hecho': 'fecha_hecho',
  'fecha_inicio': 'fecha_inicio',
  'delito': 'delito',
  'categoria': None,
  'alcaldia_hecho': 'alcaldia_hecho',
  'alcaldia_cat': None,
  'municipio': None,
  'latitud': 'latitud',
  'longitud': 'longitud'},
 {'anio': 'año',
  'entidad': 'entidad',
  'municipio': 'municipio',
  'cvegeo': None,
  'factor': 'factor',
  'mmip': 'mmip',
  'rei': 'rei',
  'nbi': 'nbi',
  'CBDj': 'CBDj',
  'CSj': 'CSj',
  'CTEU': None,
  'CENJ': 'CENJ',
  'ett': 'ett',
  'CASi': 'CASi',
  'CASSi': 'CASSi',
  'cyt': 'cyt',
  'ccevj': 'ccevj',
  'E_mmip': 'E_mmip',
  'E_rei': 'E_rei',
  'E_nbi': 'E_nbi'})

## Paso 2. Análisis y limpieza de FGJ

In [5]:
def classify_delito_block(delito_norm):
    if delito_norm is None or pd.isna(delito_norm):
        return {"is_robo_territorial": 0, "is_fraude": 0, "robo_subtipo": None}

    d = str(delito_norm)

    # Fraude
    is_fraude = 1 if "FRAUDE" in d else 0

    # Robos patrimoniales/territoriales
    cond_robo = (
        ("ROBO" in d)
        and ("ACCESORIOS DE AUTO" in d
             or "INTERIOR DE VEHICULO" in d
             or "VEHICULO" in d
             or "CASA HABITACION" in d
             or "NEGOCIO" in d
             or "TRANSEUNTE" in d
             or "REPARTIDOR" in d
             or "PASAJERO" in d
             or "TRANSPORTISTA" in d
             or "CUENTAHABIENTE" in d)
    )
    is_robo_territorial = 1 if cond_robo else 0

    subtipo = None
    if is_robo_territorial:
        if "TRANSEUNTE" in d:
            subtipo = "ROBO_A_TRANSEUNTE"
        elif "NEGOCIO" in d:
            subtipo = "ROBO_A_NEGOCIO"
        elif "CASA HABITACION" in d:
            subtipo = "ROBO_A_CASA_HABITACION"
        elif "INTERIOR DE VEHICULO" in d:
            subtipo = "ROBO_AL_INTERIOR_VEHICULO"
        elif "ACCESORIOS DE AUTO" in d:
            subtipo = "ROBO_DE_ACCESORIOS_AUTO"
        elif "VEHICULO" in d:
            subtipo = "ROBO_DE_VEHICULO"
        elif "CUENTAHABIENTE" in d:
            subtipo = "ROBO_A_CUENTAHABIENTE"
        elif "PASAJERO" in d:
            subtipo = "ROBO_A_PASAJERO"
        elif "TRANSPORTISTA" in d:
            subtipo = "ROBO_A_TRANSPORTISTA"
        elif "REPARTIDOR" in d:
            subtipo = "ROBO_A_REPARTIDOR"

    return {
        "is_robo_territorial": is_robo_territorial,
        "is_fraude": is_fraude,
        "robo_subtipo": subtipo
    }

def analyze_fgj_clean(path):
    cols = list(pd.read_csv(path, nrows=0).columns)
    total_rows = 0
    null_counts = pd.Series(0, index=cols, dtype="int64")
    delito_counter = Counter()
    categoria_counter = Counter()
    alcaldia_counter = Counter()
    anio_hecho_counter = Counter()
    anio_inicio_counter = Counter()

    sample_frames = []
    agg_chunks = []

    iter_csv, enc = read_csv_flexible(path, chunksize=FGJ_CHUNKSIZE, low_memory=False)

    for i, chunk in enumerate(iter_csv, start=1):
        print(f"[FGJ] chunk {i} | filas: {len(chunk):,}")
        total_rows += len(chunk)
        null_counts = null_counts.add(chunk.isna().sum(), fill_value=0)

        # Fechas y años
        ch = FGJ_COLS["anio_hecho"]
        ci = FGJ_COLS["anio_inicio"]
        fh = FGJ_COLS["fecha_hecho"]
        fi = FGJ_COLS["fecha_inicio"]

        if fh:
            chunk["fecha_hecho_dt"] = pd.to_datetime(chunk[fh], errors="coerce", dayfirst=True)
        else:
            chunk["fecha_hecho_dt"] = pd.NaT

        if fi:
            chunk["fecha_inicio_dt"] = pd.to_datetime(chunk[fi], errors="coerce", dayfirst=True)
        else:
            chunk["fecha_inicio_dt"] = pd.NaT

        if ch:
            chunk["anio_hecho_raw"] = pd.to_numeric(chunk[ch], errors="coerce")
        else:
            chunk["anio_hecho_raw"] = np.nan

        if ci:
            chunk["anio_inicio_raw"] = pd.to_numeric(chunk[ci], errors="coerce")
        else:
            chunk["anio_inicio_raw"] = np.nan

        # Año limpio: primero anio_hecho válido, si no, fecha_hecho, y como último respaldo fecha_inicio
        chunk["anio_hecho_clean"] = chunk["anio_hecho_raw"]
        mask_invalid = ~chunk["anio_hecho_clean"].between(2016, 2025, inclusive="both")
        chunk.loc[mask_invalid, "anio_hecho_clean"] = chunk.loc[mask_invalid, "fecha_hecho_dt"].dt.year
        mask_invalid = ~chunk["anio_hecho_clean"].between(2016, 2025, inclusive="both")
        chunk.loc[mask_invalid, "anio_hecho_clean"] = chunk.loc[mask_invalid, "fecha_inicio_dt"].dt.year
        chunk["anio_hecho_clean"] = pd.to_numeric(chunk["anio_hecho_clean"], errors="coerce")
        chunk = chunk[chunk["anio_hecho_clean"].between(2016, 2025, inclusive="both")].copy()

        if chunk.empty:
            continue

        # Texto
        if FGJ_COLS["delito"]:
            chunk["delito_norm"] = chunk[FGJ_COLS["delito"]].map(normalize_text)
        else:
            chunk["delito_norm"] = None

        if FGJ_COLS["categoria"]:
            chunk["categoria_norm"] = chunk[FGJ_COLS["categoria"]].map(normalize_text)
        else:
            chunk["categoria_norm"] = None

        if FGJ_COLS["alcaldia_hecho"]:
            chunk["alcaldia_hecho_norm"] = chunk[FGJ_COLS["alcaldia_hecho"]].map(normalize_alcaldia)
        else:
            chunk["alcaldia_hecho_norm"] = None

        if FGJ_COLS["alcaldia_cat"]:
            chunk["alcaldia_cat_norm"] = chunk[FGJ_COLS["alcaldia_cat"]].map(normalize_alcaldia)
        else:
            chunk["alcaldia_cat_norm"] = None

        chunk["alcaldia_norm"] = chunk["alcaldia_hecho_norm"].fillna(chunk["alcaldia_cat_norm"])

        # Clasificación temática de delito
        bloques = chunk["delito_norm"].apply(classify_delito_block).apply(pd.Series)
        chunk = pd.concat([chunk, bloques], axis=1)

        # Contadores
        delito_counter.update(chunk["delito_norm"].fillna("<<NA>>").value_counts().to_dict())
        categoria_counter.update(chunk["categoria_norm"].fillna("<<NA>>").value_counts().to_dict())
        alcaldia_counter.update(chunk["alcaldia_norm"].fillna("<<NA>>").value_counts().to_dict())
        anio_hecho_counter.update(chunk["anio_hecho_clean"].astype(int).value_counts().to_dict())
        anio_inicio_counter.update(chunk["anio_inicio_raw"].dropna().astype(int).value_counts().to_dict())

        # muestra
        sample_cols = [c for c in [
            FGJ_COLS["anio_hecho"], FGJ_COLS["fecha_hecho"], FGJ_COLS["anio_inicio"], FGJ_COLS["fecha_inicio"],
            FGJ_COLS["delito"], FGJ_COLS["categoria"], FGJ_COLS["alcaldia_hecho"], FGJ_COLS["alcaldia_cat"],
            FGJ_COLS["municipio"], FGJ_COLS["latitud"], FGJ_COLS["longitud"],
            "anio_hecho_clean", "delito_norm", "alcaldia_norm", "is_robo_territorial", "is_fraude", "robo_subtipo"
        ] if c is not None and c in chunk.columns]
        sample_frames.append(chunk[sample_cols].head(25))

        # agregado alcaldía-año
        grp = (
            chunk.groupby(["anio_hecho_clean", "alcaldia_norm"], dropna=False)
            .agg(
                total_carpetas=("delito_norm", "size"),
                robos_territoriales=("is_robo_territorial", "sum"),
                fraude=("is_fraude", "sum"),
                delitos_unicos=("delito_norm", "nunique"),
                categorias_unicas=("categoria_norm", "nunique"),
                pct_latitud_nula=(FGJ_COLS["latitud"], lambda s: round(s.isna().mean() * 100, 4)) if FGJ_COLS["latitud"] else ("delito_norm", lambda s: np.nan),
                pct_longitud_nula=(FGJ_COLS["longitud"], lambda s: round(s.isna().mean() * 100, 4)) if FGJ_COLS["longitud"] else ("delito_norm", lambda s: np.nan),
            )
            .reset_index()
            .rename(columns={"anio_hecho_clean": "anio"})
        )

        # subtipos de robo
        if chunk["is_robo_territorial"].sum() > 0:
            subt = (
                chunk[chunk["is_robo_territorial"] == 1]
                .groupby(["anio_hecho_clean", "alcaldia_norm", "robo_subtipo"], dropna=False)
                .size()
                .reset_index(name="n")
                .pivot_table(index=["anio_hecho_clean", "alcaldia_norm"], columns="robo_subtipo", values="n", fill_value=0)
                .reset_index()
                .rename(columns={"anio_hecho_clean": "anio"})
            )
            grp = grp.merge(subt, on=["anio", "alcaldia_norm"], how="left")

        agg_chunks.append(grp)

    fgj = pd.concat(sample_frames, ignore_index=True) if sample_frames else pd.DataFrame()
    fgj_agg = pd.concat(agg_chunks, ignore_index=True) if agg_chunks else pd.DataFrame()

    if not fgj_agg.empty:
        fgj_agg = (
            fgj_agg.groupby(["anio", "alcaldia_norm"], dropna=False)
            .sum(numeric_only=True)
            .reset_index()
            .sort_values(["anio", "alcaldia_norm"])
        )

    null_report = pd.DataFrame({
        "columna": cols,
        "nulos": [int(null_counts.get(c, 0)) for c in cols],
        "pct_nulos": [round(float(null_counts.get(c, 0)) / total_rows * 100, 4) if total_rows else np.nan for c in cols]
    }).sort_values("pct_nulos", ascending=False)

    save_csv(null_report, "fgj_null_report_global.csv")
    save_csv(pd.DataFrame(delito_counter.most_common(300), columns=["delito", "frecuencia"]), "fgj_top_delitos_clean.csv")
    save_csv(pd.DataFrame(categoria_counter.most_common(100), columns=["categoria", "frecuencia"]), "fgj_top_categorias_clean.csv")
    save_csv(pd.DataFrame(alcaldia_counter.most_common(50), columns=["alcaldia", "frecuencia"]), "fgj_top_alcaldias_clean.csv")
    save_csv(pd.DataFrame(sorted(anio_hecho_counter.items()), columns=["anio_hecho_clean", "frecuencia"]), "fgj_anio_hecho_clean_counts.csv")
    save_csv(pd.DataFrame(sorted(anio_inicio_counter.items()), columns=["anio_inicio", "frecuencia"]), "fgj_anio_inicio_counts_clean.csv")
    save_csv(fgj.head(100), "fgj_muestra_limpia.csv")
    save_csv(fgj_agg, "fgj_agregado_alcaldia_anio_clean.csv")

    resumen = {
        "fgj_filas_validas_post_clean": int(sum(anio_hecho_counter.values())),
        "fgj_anios_validos": sorted(list(anio_hecho_counter.keys())),
        "fgj_alcaldias_detectadas": int(len([k for k in alcaldia_counter if k != "<<NA>>"])),
        "share_robo_territorial_total": float(
            fgj_agg["robos_territoriales"].sum() / fgj_agg["total_carpetas"].sum()
        ) if not fgj_agg.empty and fgj_agg["total_carpetas"].sum() > 0 else np.nan,
        "share_fraude_total": float(
            fgj_agg["fraude"].sum() / fgj_agg["total_carpetas"].sum()
        ) if not fgj_agg.empty and fgj_agg["total_carpetas"].sum() > 0 else np.nan,
    }
    save_json(resumen, "fgj_resumen_clean.json")
    return fgj, fgj_agg, null_report, resumen

fgj_sample, fgj_agg, fgj_null_report, fgj_resumen = analyze_fgj_clean(FGJ_FILE)
fgj_resumen

[OK] Archivo leído con encoding: utf-8
[FGJ] chunk 1 | filas: 200,000


C:\Users\salaz\AppData\Local\Temp\ipykernel_12992\2442815962.py:82: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  chunk["fecha_hecho_dt"] = pd.to_datetime(chunk[fh], errors="coerce", dayfirst=True)


[FGJ] chunk 2 | filas: 200,000


C:\Users\salaz\AppData\Local\Temp\ipykernel_12992\2442815962.py:82: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  chunk["fecha_hecho_dt"] = pd.to_datetime(chunk[fh], errors="coerce", dayfirst=True)
C:\Users\salaz\AppData\Local\Temp\ipykernel_12992\2442815962.py:87: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  chunk["fecha_inicio_dt"] = pd.to_datetime(chunk[fi], errors="coerce", dayfirst=True)


[FGJ] chunk 3 | filas: 200,000
[FGJ] chunk 4 | filas: 200,000


...

C:\Users\salaz\AppData\Local\Temp\ipykernel_12992\2442815962.py:87: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  chunk["fecha_inicio_dt"] = pd.to_datetime(chunk[fi], errors="coerce", dayfirst=True)


[FGJ] chunk 11 | filas: 98,743


C:\Users\salaz\AppData\Local\Temp\ipykernel_12992\2442815962.py:87: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  chunk["fecha_inicio_dt"] = pd.to_datetime(chunk[fi], errors="coerce", dayfirst=True)


[OK] Guardado CSV: d:\DMProject\Documentation\eda_outputs\fgj_null_report_global.csv
[OK] Guardado CSV: d:\DMProject\Documentation\eda_outputs\fgj_top_delitos_clean.csv
[OK] Guardado CSV: d:\DMProject\Documentation\eda_outputs\fgj_top_categorias_clean.csv
[OK] Guardado CSV: d:\DMProject\Documentation\eda_outputs\fgj_top_alcaldias_clean.csv
[OK] Guardado CSV: d:\DMProject\Documentation\eda_outputs\fgj_anio_hecho_clean_counts.csv
[OK] Guardado CSV: d:\DMProject\Documentation\eda_outputs\fgj_anio_inicio_counts_clean.csv
[OK] Guardado CSV: d:\DMProject\Documentation\eda_outputs\fgj_muestra_limpia.csv
[OK] Guardado CSV: d:\DMProject\Documentation\eda_outputs\fgj_agregado_alcaldia_anio_clean.csv
[OK] Guardado JSON: d:\DMProject\Documentation\eda_outputs\fgj_resumen_clean.json


{'fgj_filas_validas_post_clean': 2086791,
 'fgj_anios_validos': [2016,
  2017,
  2018,
  2019,
  2020,
  2021,
  2022,
  2023,
  2024,
  2025],
 'fgj_alcaldias_detectadas': 17,
 'share_robo_territorial_total': 0.30406494948463936,
 'share_fraude_total': 0.07337294439165207}

### Vista rápida de FGJ limpio

In [6]:
display(pd.read_csv(OUT_DIR / "fgj_anio_hecho_clean_counts.csv").head(20))
display(pd.read_csv(OUT_DIR / "fgj_top_delitos_clean.csv").head(25))
display(pd.read_csv(OUT_DIR / "fgj_top_categorias_clean.csv").head(20))
display(pd.read_csv(OUT_DIR / "fgj_top_alcaldias_clean.csv").head(20))
display(fgj_agg.head(20))

,anio_hecho_clean,frecuencia
0,2016,204546
1,2017,232371
2,2018,257161
3,2019,248693
4,2020,207119
5,2021,229158
6,2022,239059
7,2023,238466
8,2024,217702
9,2025,12516


,delito,frecuencia
0,VIOLENCIA FAMILIAR,261172
1,FRAUDE,152837
2,AMENAZAS,135603
3,ROBO DE OBJETOS,112205
4,ROBO A TRANSEUNTE EN VIA PUBLICA CON VIOLENCIA,88453
5,ROBO A NEGOCIO SIN VIOLENCIA,75872
6,ROBO DE ACCESORIOS DE AUTO,74457
7,ROBO DE OBJETOS DEL INTERIOR DE UN VEHICULO,57559
8,ROBO DE VEHICULO DE SERVICIO PARTICULAR SIN VI...,42161
9,ROBO A CASA HABITACION SIN VIOLENCIA,40794


,categoria,frecuencia
0,<<NA>>,2086791


,alcaldia,frecuencia
0,CUAUHTEMOC,315194
1,IZTAPALAPA,303289
2,GUSTAVO A MADERO,208188
3,BENITO JUAREZ,163393
4,COYOACAN,141313
5,ALVARO OBREGON,139461
6,MIGUEL HIDALGO,129648
7,TLALPAN,125203
8,VENUSTIANO CARRANZA,118296
9,AZCAPOTZALCO,98536


,anio,alcaldia_norm,total_carpetas,robos_territoriales,fraude,delitos_unicos,categorias_unicas,pct_latitud_nula,pct_longitud_nula,ROBO_A_CASA_HABITACION,ROBO_A_NEGOCIO,ROBO_A_PASAJERO,ROBO_A_REPARTIDOR,ROBO_A_TRANSEUNTE,ROBO_DE_ACCESORIOS_AUTO,ROBO_DE_VEHICULO,ROBO_A_TRANSPORTISTA
0,2016.0,ALVARO OBREGON,12823,3993.0,650.0,282,0,16.3728,16.3728,366.0,1068.0,134.0,110.0,932.0,223.0,1160.0,0.0
1,2016.0,AZCAPOTZALCO,10455,3852.0,516.0,274,0,24.5944,24.5944,371.0,1021.0,124.0,85.0,802.0,288.0,1161.0,0.0
2,2016.0,BENITO JUAREZ,17346,6049.0,1757.0,327,0,4.6731,4.6731,647.0,1776.0,194.0,49.0,1265.0,679.0,1439.0,0.0
3,2016.0,CDMX INDETERMINADA,41,0.0,6.0,27,0,600.0000,600.0000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2016.0,COYOACAN,14204,4697.0,826.0,292,0,27.2498,27.2498,480.0,1348.0,129.0,45.0,875.0,344.0,1476.0,0.0
5,2016.0,CUAJIMALPA DE MORELOS,2972,641.0,204.0,206,0,25.0915,25.0915,91.0,280.0,12.0,10.0,74.0,29.0,145.0,0.0
6,2016.0,CUAUHTEMOC,34214,9057.0,3573.0,448,0,154.7850,154.7850,566.0,2440.0,827.0,112.0,3192.0,503.0,1417.0,0.0
7,2016.0,GUSTAVO A MADERO,19754,6984.0,864.0,340,0,48.5220,48.5220,673.0,1768.0,344.0,241.0,1535.0,353.0,2070.0,0.0
8,2016.0,IZTACALCO,8408,2524.0,315.0,246,0,20.6091,20.6091,282.0,393.0,157.0,61.0,676.0,185.0,770.0,0.0
9,2016.0,IZTAPALAPA,30323,10331.0,1012.0,371,0,51.1600,51.1600,891.0,2611.0,521.0,436.0,2806.0,342.0,2724.0,0.0


## Paso 3. Análisis y filtrado territorial correcto de pobreza

In [7]:
def analyze_pobreza_clean(path):
    pobreza, enc = read_csv_flexible(path, low_memory=False)
    cols = pobreza.columns.tolist()

    col_anio = POB_COLS["anio"]
    col_ent = POB_COLS["entidad"]
    col_mun = POB_COLS["municipio"]
    col_cve = POB_COLS["cvegeo"]
    col_factor = POB_COLS["factor"]

    # Nulos
    null_report = pd.DataFrame({
        "columna": cols,
        "dtype": pobreza.dtypes.astype(str).values,
        "nulos": pobreza.isna().sum().values,
        "pct_nulos": (pobreza.isna().sum().values / len(pobreza) * 100).round(4),
        "n_unicos_sin_na": [pobreza[c].nunique(dropna=True) for c in cols]
    }).sort_values("pct_nulos", ascending=False)
    save_csv(null_report, "pobreza_null_report_global.csv")

    # Años
    if col_anio:
        pobreza["anio"] = pd.to_numeric(pobreza[col_anio], errors="coerce")
    else:
        pobreza["anio"] = np.nan

    # Detectar CDMX
    pobreza["is_cdmx"] = False

    if col_ent:
        ent_num = pd.to_numeric(pobreza[col_ent], errors="coerce")
        pobreza.loc[ent_num == 9, "is_cdmx"] = True

    if col_cve:
        cve_str = pobreza[col_cve].astype(str).str.strip()
        pobreza.loc[cve_str.str.startswith("09", na=False), "is_cdmx"] = True

    # Si no quedó nada marcado, conservamos todo pero lo reportamos
    if pobreza["is_cdmx"].sum() == 0:
        print("[ADVERTENCIA] No se pudo filtrar CDMX automáticamente con entidad/cvegeo. Se conserva toda la base.")
        pobreza["is_cdmx"] = True

    pobreza_cdmx = pobreza[pobreza["is_cdmx"]].copy()

    # Mapear alcaldía solo después del filtrado a CDMX
    if col_mun:
        pobreza_cdmx["municipio_num"] = pd.to_numeric(pobreza_cdmx[col_mun], errors="coerce")
        pobreza_cdmx["alcaldia_norm"] = pobreza_cdmx["municipio_num"].map(MAP_MUNICIPIO_CDMX).map(normalize_alcaldia)
    else:
        pobreza_cdmx["municipio_num"] = np.nan
        pobreza_cdmx["alcaldia_norm"] = None

    # Métricas de verificación territorial
    entidad_counts = pd.DataFrame()
    if col_ent:
        entidad_counts = pobreza[col_ent].value_counts(dropna=False).reset_index()
        entidad_counts.columns = ["entidad", "frecuencia"]
        save_csv(entidad_counts, "pobreza_entidad_counts.csv")

    municipio_counts = pd.DataFrame()
    if col_mun:
        municipio_counts = pobreza[col_mun].value_counts(dropna=False).reset_index()
        municipio_counts.columns = ["municipio", "frecuencia"]
        save_csv(municipio_counts, "pobreza_municipio_counts.csv")

    municipio_counts_cdmx = pd.DataFrame()
    if col_mun:
        municipio_counts_cdmx = pobreza_cdmx[col_mun].value_counts(dropna=False).reset_index()
        municipio_counts_cdmx.columns = ["municipio_cdmx", "frecuencia"]
        save_csv(municipio_counts_cdmx, "pobreza_municipio_counts_cdmx.csv")

    save_csv(pobreza_cdmx.head(200), "pobreza_cdmx_muestra.csv")

    # Agregado alcaldía-año
    cont_map = {
        "mmip": POB_COLS["mmip"],
        "rei": POB_COLS["rei"],
        "nbi": POB_COLS["nbi"],
        "CBDj": POB_COLS["CBDj"],
        "CSj": POB_COLS["CSj"],
        "CTEU": POB_COLS["CTEU"],
        "CENJ": POB_COLS["CENJ"],
        "ett": POB_COLS["ett"],
        "CASi": POB_COLS["CASi"],
        "CASSi": POB_COLS["CASSi"],
        "cyt": POB_COLS["cyt"],
        "ccevj": POB_COLS["ccevj"],
    }
    cont_map = {k: v for k, v in cont_map.items() if v is not None}

    estr_map = {
        "E_mmip": POB_COLS["E_mmip"],
        "E_rei": POB_COLS["E_rei"],
        "E_nbi": POB_COLS["E_nbi"],
    }
    estr_map = {k: v for k, v in estr_map.items() if v is not None}

    if col_factor is None:
        raise ValueError("No se encontró la columna 'factor' en la base de pobreza; se necesita para agregación ponderada.")

    pobreza_cdmx[col_factor] = pd.to_numeric(pobreza_cdmx[col_factor], errors="coerce")

    rows = []
    for (anio, alcaldia), sub in pobreza_cdmx.groupby(["anio", "alcaldia_norm"], dropna=False):
        row = {
            "anio": int(anio) if pd.notna(anio) else np.nan,
            "alcaldia_norm": alcaldia,
            "n_registros": int(len(sub)),
            "factor_sum": float(pd.to_numeric(sub[col_factor], errors="coerce").sum(skipna=True)),
        }
        for k, col_real in cont_map.items():
            row[f"{k}_wmean"] = weighted_mean(sub[col_real], sub[col_factor])

        for k, col_real in estr_map.items():
            row[f"{k}_wmode"] = weighted_mode(sub[col_real], sub[col_factor])

        rows.append(row)

    pobreza_agg = pd.DataFrame(rows).sort_values(["anio", "alcaldia_norm"]).reset_index(drop=True)

    # Frecuencias
    anio_counts = pobreza_cdmx["anio"].dropna().astype(int).value_counts().sort_index().reset_index()
    anio_counts.columns = ["anio", "frecuencia"]
    save_csv(anio_counts, "pobreza_anio_counts_cdmx.csv")
    save_csv(pobreza_agg, "pobreza_agregado_alcaldia_anio_clean.csv")

    resumen = {
        "pobreza_total_filas": int(len(pobreza)),
        "pobreza_filas_cdmx": int(len(pobreza_cdmx)),
        "share_cdmx_sobre_total": float(len(pobreza_cdmx) / len(pobreza)) if len(pobreza) else np.nan,
        "anios_cdmx_detectados": sorted(pobreza_cdmx["anio"].dropna().astype(int).unique().tolist()),
        "n_alcaldias_mapeadas_cdmx": int(pobreza_cdmx["alcaldia_norm"].nunique(dropna=True)),
        "municipios_distintos_cdmx": sorted(pobreza_cdmx["municipio_num"].dropna().astype(int).unique().tolist()) if col_mun else [],
        "variables_continuas_usadas": list(cont_map.keys()),
        "variables_estrato_usadas": list(estr_map.keys()),
        "encoding_usado": enc,
    }
    save_json(resumen, "pobreza_resumen_clean.json")
    return pobreza_cdmx, pobreza_agg, null_report, resumen

pobreza_cdmx, pobreza_agg, pobreza_null_report, pobreza_resumen = analyze_pobreza_clean(POBREZA_FILE)
pobreza_resumen

[OK] Archivo leído con encoding: cp1252
[OK] Guardado CSV: d:\DMProject\Documentation\eda_outputs\pobreza_null_report_global.csv
[OK] Guardado CSV: d:\DMProject\Documentation\eda_outputs\pobreza_entidad_counts.csv
[OK] Guardado CSV: d:\DMProject\Documentation\eda_outputs\pobreza_municipio_counts.csv
[OK] Guardado CSV: d:\DMProject\Documentation\eda_outputs\pobreza_municipio_counts_cdmx.csv
[OK] Guardado CSV: d:\DMProject\Documentation\eda_outputs\pobreza_cdmx_muestra.csv
[OK] Guardado CSV: d:\DMProject\Documentation\eda_outputs\pobreza_anio_counts_cdmx.csv
[OK] Guardado CSV: d:\DMProject\Documentation\eda_outputs\pobreza_agregado_alcaldia_anio_clean.csv
[OK] Guardado JSON: d:\DMProject\Documentation\eda_outputs\pobreza_resumen_clean.json


{'pobreza_total_filas': 842342,
 'pobreza_filas_cdmx': 22219,
 'share_cdmx_sobre_total': 0.02637764708396352,
 'anios_cdmx_detectados': [2016, 2018, 2020],
 'n_alcaldias_mapeadas_cdmx': 16,
 'municipios_distintos_cdmx': [2,
  3,
  4,
  5,
  6,
  7,
  8,
  9,
  10,
  11,
  12,
  13,
  14,
  15,
  16,
  17],
 'variables_continuas_usadas': ['mmip',
  'rei',
  'nbi',
  'CBDj',
  'CSj',
  'CENJ',
  'ett',
  'CASi',
  'CASSi',
  'cyt',
  'ccevj'],
 'variables_estrato_usadas': ['E_mmip', 'E_rei', 'E_nbi'],
 'encoding_usado': 'cp1252'}

### Vista rápida de pobreza limpia

In [8]:
display(pd.read_csv(OUT_DIR / "pobreza_entidad_counts.csv").head(20) if (OUT_DIR / "pobreza_entidad_counts.csv").exists() else pd.DataFrame())
display(pd.read_csv(OUT_DIR / "pobreza_municipio_counts_cdmx.csv").head(25) if (OUT_DIR / "pobreza_municipio_counts_cdmx.csv").exists() else pd.DataFrame())
display(pd.read_csv(OUT_DIR / "pobreza_anio_counts_cdmx.csv"))
display(pobreza_agg.head(20))

,entidad,frecuencia
0,8,38109
1,2,37771
2,5,37297
3,15,35682
4,22,33813
5,19,32985
6,25,32868
7,11,32097
8,6,29072
9,31,28468


,municipio_cdmx,frecuencia
0,7,3545
1,5,2336
2,12,2260
3,9,1972
4,10,1624
5,13,1563
6,15,1266
7,3,1193
8,11,1092
9,17,1000


,anio,frecuencia
0,2016,5714
1,2018,7462
2,2020,9043


,anio,alcaldia_norm,n_registros,factor_sum,mmip_wmean,rei_wmean,nbi_wmean,CBDj_wmean,CSj_wmean,CENJ_wmean,ett_wmean,CASi_wmean,CASSi_wmean,cyt_wmean,ccevj_wmean,E_mmip_wmode,E_rei_wmode,E_nbi_wmode
0,2016,ALVARO OBREGON,397,659539.0,-0.016351,-0.095012,-0.020541,-0.449621,0.067644,0.000000,0.978426,0.083492,0.360959,-0.013848,-0.085989,3,5,5
1,2016,AZCAPOTZALCO,245,393527.0,-0.006374,-0.069260,-0.018415,-0.457162,0.087811,0.005033,1.000432,0.097185,0.305492,0.000820,-0.083150,3,5,5
2,2016,BENITO JUAREZ,242,525216.0,-0.320988,-0.249439,-0.212123,-0.708901,0.020899,0.000000,0.962998,-0.150982,0.201860,-0.386028,-0.325615,5,5,5
3,2016,COYOACAN,277,459883.0,0.010757,-0.081690,0.029540,-0.304103,0.129663,0.000000,0.964209,0.102278,0.381430,-0.000466,0.008096,3,5,3
4,2016,CUAJIMALPA DE MORELOS,169,216080.0,0.097752,0.009551,0.116836,-0.225012,0.225122,0.009180,1.031619,0.129088,0.285255,0.086351,0.200665,3,4,3
5,2016,CUAUHTEMOC,589,951496.0,-0.035068,-0.130129,-0.074666,-0.349289,0.021946,0.000307,1.077736,0.042211,0.327553,-0.011411,-0.198367,5,5,5
6,2016,GUSTAVO A MADERO,672,1016468.0,0.113504,-0.026261,0.051609,-0.326945,0.114592,0.000934,1.040957,0.161293,0.410895,0.150483,-0.000690,3,5,3
7,2016,IZTACALCO,286,458083.0,0.072060,-0.099846,0.034347,-0.409135,0.106549,0.006112,1.106485,0.162212,0.414755,0.094591,0.017140,3,5,3
8,2016,IZTAPALAPA,1080,1667333.0,0.187446,0.009722,0.134874,-0.299521,0.247083,0.001141,1.041733,0.193418,0.538850,0.218855,0.133885,3,4,3
9,2016,LA MAGDALENA CONTRERAS,81,133845.0,0.018614,-0.091742,0.023623,-0.454331,0.131305,0.000000,0.874903,0.051906,0.378468,0.015621,0.043135,3,5,5


## Paso 4. Construcción de panel contemporáneo y panel rezagado

In [9]:
# Años comunes
fgj_years = sorted(pd.to_numeric(fgj_agg["anio"], errors="coerce").dropna().astype(int).unique().tolist())
pob_years = sorted(pd.to_numeric(pobreza_agg["anio"], errors="coerce").dropna().astype(int).unique().tolist())
common_years = sorted(set(fgj_years).intersection(set(pob_years)))
lag_years = [y for y in fgj_years if len(common_years) > 0 and y > max(common_years)]

# Join contemporáneo
panel_contemporaneo = pd.merge(
    pobreza_agg[pobreza_agg["anio"].isin(common_years)],
    fgj_agg[fgj_agg["anio"].isin(common_years)],
    on=["anio", "alcaldia_norm"],
    how="inner"
).sort_values(["anio", "alcaldia_norm"]).reset_index(drop=True)

# Base social promedio del periodo común
social_cols = [c for c in pobreza_agg.columns if c.endswith("_wmean")]
social_mode_cols = [c for c in pobreza_agg.columns if c.endswith("_wmode")]

base_social = (
    pobreza_agg[pobreza_agg["anio"].isin(common_years)]
    .groupby("alcaldia_norm", dropna=False)[social_cols]
    .mean(numeric_only=True)
    .reset_index()
)

# Join rezagado
panel_rezagado = pd.merge(
    fgj_agg[fgj_agg["anio"].isin(lag_years)],
    base_social,
    on="alcaldia_norm",
    how="left"
).sort_values(["anio", "alcaldia_norm"]).reset_index(drop=True)

save_csv(panel_contemporaneo, "panel_contemporaneo.csv")
save_csv(panel_rezagado, "panel_rezagado.csv")

evaluacion_temporalidad = {
    "fgj_years": fgj_years,
    "pobreza_years": pob_years,
    "common_years": common_years,
    "lag_years": lag_years,
    "alcaldias_comunes_contemporaneo": sorted(panel_contemporaneo["alcaldia_norm"].dropna().unique().tolist()) if not panel_contemporaneo.empty else [],
    "n_obs_panel_contemporaneo": int(len(panel_contemporaneo)),
    "n_obs_panel_rezagado": int(len(panel_rezagado)),
}
save_json(evaluacion_temporalidad, "evaluacion_temporalidad_corregida.json")
evaluacion_temporalidad

[OK] Guardado CSV: d:\DMProject\Documentation\eda_outputs\panel_contemporaneo.csv
[OK] Guardado CSV: d:\DMProject\Documentation\eda_outputs\panel_rezagado.csv
[OK] Guardado JSON: d:\DMProject\Documentation\eda_outputs\evaluacion_temporalidad_corregida.json


{'fgj_years': [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025],
 'pobreza_years': [2016, 2018, 2020],
 'common_years': [2016, 2018, 2020],
 'lag_years': [2021, 2022, 2023, 2024, 2025],
 'alcaldias_comunes_contemporaneo': ['ALVARO OBREGON',
  'AZCAPOTZALCO',
  'BENITO JUAREZ',
  'COYOACAN',
  'CUAJIMALPA DE MORELOS',
  'CUAUHTEMOC',
  'GUSTAVO A MADERO',
  'IZTACALCO',
  'IZTAPALAPA',
  'LA MAGDALENA CONTRERAS',
  'MIGUEL HIDALGO',
  'MILPA ALTA',
  'TLAHUAC',
  'TLALPAN',
  'VENUSTIANO CARRANZA',
  'XOCHIMILCO'],
 'n_obs_panel_contemporaneo': 48,
 'n_obs_panel_rezagado': 90}

## Paso 5. Métricas reportables y asociaciones iniciales

In [10]:
def add_rates(df):
    out = df.copy()
    if "total_carpetas" in out.columns:
        out["share_robos_territoriales"] = out["robos_territoriales"] / out["total_carpetas"].replace(0, np.nan)
        out["share_fraude"] = out["fraude"] / out["total_carpetas"].replace(0, np.nan)
    return out

panel_contemporaneo = add_rates(panel_contemporaneo)
panel_rezagado = add_rates(panel_rezagado)

# Correlaciones relevantes (solo numéricas)
def top_correlations(df, target, min_abs=0.2):
    num = df.select_dtypes(include=[np.number]).copy()
    if target not in num.columns or num[target].notna().sum() < 3:
        return pd.DataFrame(columns=["variable", "target", "corr"])
    corrs = num.corr(numeric_only=True)[target].drop(target).dropna().sort_values(key=lambda s: s.abs(), ascending=False)
    out = corrs.reset_index()
    out.columns = ["variable", "corr"]
    out["target"] = target
    out = out[out["corr"].abs() >= min_abs]
    return out[["variable", "target", "corr"]]

corr_robos_contemp = top_correlations(panel_contemporaneo, "robos_territoriales", min_abs=0.15)
corr_fraude_contemp = top_correlations(panel_contemporaneo, "fraude", min_abs=0.15)
corr_robos_lag = top_correlations(panel_rezagado, "robos_territoriales", min_abs=0.15)
corr_fraude_lag = top_correlations(panel_rezagado, "fraude", min_abs=0.15)

save_csv(corr_robos_contemp, "top_asociaciones_robos_territoriales_contemporaneo.csv")
save_csv(corr_fraude_contemp, "top_asociaciones_fraude_contemporaneo.csv")
save_csv(corr_robos_lag, "top_asociaciones_robos_territoriales_rezagado.csv")
save_csv(corr_fraude_lag, "top_asociaciones_fraude_rezagado.csv")

resumen_metricas = {
    "fgj_filas_limpias": int(fgj_resumen["fgj_filas_validas_post_clean"]),
    "fgj_anios_validos": sorted(pd.to_numeric(fgj_agg["anio"], errors="coerce").dropna().astype(int).unique().tolist()),
    "pobreza_filas_cdmx": int(pobreza_resumen["pobreza_filas_cdmx"]),
    "pobreza_anios_cdmx": pobreza_resumen["anios_cdmx_detectados"],
    "panel_contemporaneo_filas": int(len(panel_contemporaneo)),
    "panel_rezagado_filas": int(len(panel_rezagado)),
    "alcaldias_panel_contemporaneo": int(panel_contemporaneo["alcaldia_norm"].nunique()) if not panel_contemporaneo.empty else 0,
    "alcaldias_panel_rezagado": int(panel_rezagado["alcaldia_norm"].nunique()) if not panel_rezagado.empty else 0,
    "share_robos_territoriales_sobre_fgj": float(fgj_agg["robos_territoriales"].sum() / fgj_agg["total_carpetas"].sum()) if not fgj_agg.empty and fgj_agg["total_carpetas"].sum() > 0 else np.nan,
    "share_fraude_sobre_fgj": float(fgj_agg["fraude"].sum() / fgj_agg["total_carpetas"].sum()) if not fgj_agg.empty and fgj_agg["total_carpetas"].sum() > 0 else np.nan,
    "common_years": common_years,
    "lag_years": lag_years,
}
save_json(resumen_metricas, "resumen_metricas_proyecto.json")
resumen_metricas

[OK] Guardado CSV: d:\DMProject\Documentation\eda_outputs\top_asociaciones_robos_territoriales_contemporaneo.csv
[OK] Guardado CSV: d:\DMProject\Documentation\eda_outputs\top_asociaciones_fraude_contemporaneo.csv
[OK] Guardado CSV: d:\DMProject\Documentation\eda_outputs\top_asociaciones_robos_territoriales_rezagado.csv
[OK] Guardado CSV: d:\DMProject\Documentation\eda_outputs\top_asociaciones_fraude_rezagado.csv
[OK] Guardado JSON: d:\DMProject\Documentation\eda_outputs\resumen_metricas_proyecto.json


{'fgj_filas_limpias': 2086791,
 'fgj_anios_validos': [2016,
  2017,
  2018,
  2019,
  2020,
  2021,
  2022,
  2023,
  2024,
  2025],
 'pobreza_filas_cdmx': 22219,
 'pobreza_anios_cdmx': [2016, 2018, 2020],
 'panel_contemporaneo_filas': 48,
 'panel_rezagado_filas': 90,
 'alcaldias_panel_contemporaneo': 16,
 'alcaldias_panel_rezagado': 17,
 'share_robos_territoriales_sobre_fgj': 0.30406494948463936,
 'share_fraude_sobre_fgj': 0.07337294439165207,
 'common_years': [2016, 2018, 2020],
 'lag_years': [2021, 2022, 2023, 2024, 2025]}

In [11]:
display(corr_robos_contemp.head(20))
display(corr_fraude_contemp.head(20))
display(corr_robos_lag.head(20))
display(corr_fraude_lag.head(20))

,variable,target,corr
0,total_carpetas,robos_territoriales,0.964449
1,ROBO_A_NEGOCIO,robos_territoriales,0.964298
2,ROBO_A_TRANSEUNTE,robos_territoriales,0.954294
3,ROBO_DE_VEHICULO,robos_territoriales,0.937411
4,delitos_unicos,robos_territoriales,0.911358
5,ROBO_A_CASA_HABITACION,robos_territoriales,0.870687
6,ROBO_A_PASAJERO,robos_territoriales,0.832447
7,fraude,robos_territoriales,0.742261
8,ROBO_DE_ACCESORIOS_AUTO,robos_territoriales,0.732625
9,ROBO_A_REPARTIDOR,robos_territoriales,0.706494


,variable,target,corr
0,total_carpetas,fraude,0.811780
1,ROBO_A_NEGOCIO,fraude,0.788451
2,ROBO_DE_ACCESORIOS_AUTO,fraude,0.779627
3,robos_territoriales,fraude,0.742261
4,delitos_unicos,fraude,0.724532
5,ROBO_A_PASAJERO,fraude,0.711304
6,share_fraude,fraude,0.698063
7,ROBO_A_TRANSEUNTE,fraude,0.674434
8,pct_latitud_nula,fraude,0.639856
9,pct_longitud_nula,fraude,0.639856


,variable,target,corr
0,total_carpetas,robos_territoriales,0.975796
1,ROBO_A_NEGOCIO,robos_territoriales,0.970491
2,ROBO_DE_VEHICULO,robos_territoriales,0.964968
3,ROBO_A_TRANSEUNTE,robos_territoriales,0.948760
4,fraude,robos_territoriales,0.915859
5,ROBO_DE_ACCESORIOS_AUTO,robos_territoriales,0.868907
6,ROBO_A_CASA_HABITACION,robos_territoriales,0.855369
7,ROBO_A_PASAJERO,robos_territoriales,0.844716
8,delitos_unicos,robos_territoriales,0.753249
9,ROBO_A_REPARTIDOR,robos_territoriales,0.665461


,variable,target,corr
0,ROBO_DE_ACCESORIOS_AUTO,fraude,0.928661
1,robos_territoriales,fraude,0.915859
2,ROBO_A_NEGOCIO,fraude,0.890557
3,total_carpetas,fraude,0.888088
4,ROBO_DE_VEHICULO,fraude,0.859146
5,ROBO_A_PASAJERO,fraude,0.846679
6,ROBO_A_TRANSEUNTE,fraude,0.804291
7,delitos_unicos,fraude,0.690881
8,ROBO_A_CASA_HABITACION,fraude,0.654292
9,share_fraude,fraude,0.641898


## Paso 6. Tipologías con K-means (exploratorio, no principal)

In [12]:
def kmeans_typology(df, feature_cols, out_name_prefix="panel"):
    work = df.copy()
    work = work.dropna(subset=feature_cols).copy()
    if len(work) < 9:
        print(f"[INFO] Muy pocas filas para K-means en {out_name_prefix}.")
        return pd.DataFrame()

    X = work[feature_cols].astype(float)
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)

    best_k = None
    best_score = -999
    best_labels = None

    for k in [2, 3, 4]:
        if len(work) <= k:
            continue
        model = KMeans(n_clusters=k, n_init=20, random_state=42)
        labels = model.fit_predict(Xs)
        if len(set(labels)) < 2:
            continue
        score = silhouette_score(Xs, labels)
        if score > best_score:
            best_score = score
            best_k = k
            best_labels = labels

    if best_labels is None:
        print(f"[INFO] No fue posible encontrar clusters útiles para {out_name_prefix}.")
        return pd.DataFrame()

    work["cluster"] = best_labels
    summary = work.groupby("cluster")[feature_cols].mean().reset_index()
    summary["n"] = work["cluster"].value_counts().sort_index().values

    save_csv(work, f"{out_name_prefix}_kmeans_assignments.csv")
    save_csv(summary, f"{out_name_prefix}_kmeans_summary.csv")
    save_json({"best_k": best_k, "silhouette": best_score, "features": feature_cols}, f"{out_name_prefix}_kmeans_meta.json")
    return summary

feature_cols_contemp = [c for c in ["mmip_wmean", "rei_wmean", "nbi_wmean", "robos_territoriales", "fraude"] if c in panel_contemporaneo.columns]
kmeans_summary = kmeans_typology(panel_contemporaneo, feature_cols_contemp, out_name_prefix="panel_contemporaneo")
kmeans_summary

[OK] Guardado CSV: d:\DMProject\Documentation\eda_outputs\panel_contemporaneo_kmeans_assignments.csv
[OK] Guardado CSV: d:\DMProject\Documentation\eda_outputs\panel_contemporaneo_kmeans_summary.csv
[OK] Guardado JSON: d:\DMProject\Documentation\eda_outputs\panel_contemporaneo_kmeans_meta.json


,cluster,mmip_wmean,rei_wmean,nbi_wmean,robos_territoriales,fraude,n
0,0,-0.060812,-0.135616,-0.072125,7136.411765,1624.352941,17
1,1,0.139793,-0.027492,0.083027,3480.548387,472.645161,31


## Paso 7. Predicción exploratoria del nivel de robos territoriales

Este bloque **no es el eje principal**. Solo estima si la base social permite clasificar una alcaldía-año en nivel bajo/medio/alto de robos territoriales, preferentemente en el panel rezagado.

In [13]:
def run_predictive_plus(df, feature_cols, target_col="robos_territoriales", out_prefix="predictivo_rezagado"):
    work = df.copy().dropna(subset=feature_cols + [target_col]).copy()
    if len(work) < 24:
        print("[INFO] Muy pocas observaciones para predicción exploratoria robusta.")
        return None

    # Variable objetivo categórica por terciles
    work["nivel_objetivo"] = pd.qcut(work[target_col], q=3, labels=["bajo", "medio", "alto"], duplicates="drop")
    work = work.dropna(subset=["nivel_objetivo"]).copy()

    if work["nivel_objetivo"].nunique() < 2:
        print("[INFO] La variable objetivo no tiene suficientes niveles.")
        return None

    X = work[feature_cols].astype(float)
    y = work["nivel_objetivo"].astype(str)

    # CV estratificada conservadora
    min_class = y.value_counts().min()
    n_splits = int(min(5, min_class))
    if n_splits < 2:
        print("[INFO] No hay suficientes observaciones por clase para validación cruzada.")
        return None

    model = RandomForestClassifier(
        n_estimators=300,
        max_depth=4,
        min_samples_leaf=2,
        random_state=42,
        class_weight="balanced"
    )

    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    y_pred = cross_val_predict(model, X, y, cv=cv)

    metrics = {
        "n_obs": int(len(work)),
        "n_splits": int(n_splits),
        "classes": sorted(y.unique().tolist()),
        "accuracy": float(accuracy_score(y, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y, y_pred)),
    }

    report = pd.DataFrame(classification_report(y, y_pred, output_dict=True)).T.reset_index().rename(columns={"index": "label"})
    cm = pd.DataFrame(confusion_matrix(y, y_pred, labels=sorted(y.unique())), index=[f"real_{c}" for c in sorted(y.unique())], columns=[f"pred_{c}" for c in sorted(y.unique())])

    model.fit(X, y)
    importances = pd.DataFrame({"feature": feature_cols, "importance": model.feature_importances_}).sort_values("importance", ascending=False)

    save_json(metrics, f"{out_prefix}_metrics.json")
    save_csv(report, f"{out_prefix}_classification_report.csv")
    save_csv(cm.reset_index().rename(columns={"index":"row"}), f"{out_prefix}_confusion_matrix.csv")
    save_csv(importances, f"{out_prefix}_feature_importances.csv")
    save_csv(work.assign(pred_nivel=y_pred), f"{out_prefix}_predicciones.csv")

    return metrics, report, cm, importances

feature_cols_pred = [c for c in ["mmip_wmean", "rei_wmean", "nbi_wmean", "CBDj_wmean", "CSj_wmean", "CTEU_wmean", "CENJ_wmean", "ett_wmean", "CASi_wmean", "CASSi_wmean", "cyt_wmean"] if c in panel_rezagado.columns]
pred_result = run_predictive_plus(panel_rezagado, feature_cols_pred, target_col="robos_territoriales", out_prefix="predictivo_rezagado_robos")
pred_result[0] if pred_result is not None else None

[OK] Guardado JSON: d:\DMProject\Documentation\eda_outputs\predictivo_rezagado_robos_metrics.json
[OK] Guardado CSV: d:\DMProject\Documentation\eda_outputs\predictivo_rezagado_robos_classification_report.csv
[OK] Guardado CSV: d:\DMProject\Documentation\eda_outputs\predictivo_rezagado_robos_confusion_matrix.csv
[OK] Guardado CSV: d:\DMProject\Documentation\eda_outputs\predictivo_rezagado_robos_feature_importances.csv
[OK] Guardado CSV: d:\DMProject\Documentation\eda_outputs\predictivo_rezagado_robos_predicciones.csv


{'n_obs': 80,
 'n_splits': 5,
 'classes': ['alto', 'bajo', 'medio'],
 'accuracy': 0.7,
 'balanced_accuracy': 0.6999050332383666}

## Paso 8. Diagnóstico final automático

In [14]:
diagnostico_lines = []
diagnostico_lines.append("DIAGNÓSTICO FINAL DEL NOTEBOOK\n")
diagnostico_lines.append("="*80 + "\n\n")

diagnostico_lines.append("1) Calidad y cobertura de FGJ\n")
diagnostico_lines.append(f"- Filas válidas posteriores a limpieza: {fgj_resumen['fgj_filas_validas_post_clean']:,}\n")
diagnostico_lines.append(f"- Años válidos de FGJ: {fgj_resumen['fgj_anios_validos']}\n")
diagnostico_lines.append(f"- Alcaldías detectadas en FGJ: {fgj_resumen['fgj_alcaldias_detectadas']}\n")
diagnostico_lines.append(f"- Participación de robos territoriales sobre el total FGJ: {resumen_metricas['share_robos_territoriales_sobre_fgj']:.4f}\n")
diagnostico_lines.append(f"- Participación de fraude sobre el total FGJ: {resumen_metricas['share_fraude_sobre_fgj']:.4f}\n\n")

diagnostico_lines.append("2) Calidad y cobertura de pobreza\n")
diagnostico_lines.append(f"- Filas totales pobreza: {pobreza_resumen['pobreza_total_filas']:,}\n")
diagnostico_lines.append(f"- Filas CDMX detectadas: {pobreza_resumen['pobreza_filas_cdmx']:,}\n")
diagnostico_lines.append(f"- Share CDMX sobre pobreza total: {pobreza_resumen['share_cdmx_sobre_total']:.4f}\n")
diagnostico_lines.append(f"- Años detectados en pobreza CDMX: {pobreza_resumen['anios_cdmx_detectados']}\n")
diagnostico_lines.append(f"- Alcaldías mapeadas en pobreza CDMX: {pobreza_resumen['n_alcaldias_mapeadas_cdmx']}\n\n")

diagnostico_lines.append("3) Temporalidad utilizable\n")
diagnostico_lines.append(f"- Años comunes contemporáneos: {common_years}\n")
diagnostico_lines.append(f"- Años rezagados utilizables: {lag_years}\n")
diagnostico_lines.append(f"- Filas panel contemporáneo: {len(panel_contemporaneo)}\n")
diagnostico_lines.append(f"- Filas panel rezagado: {len(panel_rezagado)}\n\n")

diagnostico_lines.append("4) Recomendación metodológica\n")
if len(common_years) >= 2:
    diagnostico_lines.append(
        f"- Ventana principal recomendada: {common_years}, con análisis explicativo por alcaldía-año.\n"
    )
else:
    diagnostico_lines.append(
        "- La ventana contemporánea es muy reducida; el estudio debería apoyarse sobre todo en la lógica rezagada.\n"
    )

if len(lag_years) >= 1:
    diagnostico_lines.append(
        f"- Ventana secundaria recomendada: {lag_years}, usando pobreza promedio del periodo común para explorar persistencia sobre FGJ posterior.\n"
    )

diagnostico_lines.append(
    "- Outcome principal sugerido: robos territoriales/patrimoniales agregados por alcaldía-año.\n"
)
diagnostico_lines.append(
    "- Outcome secundario sugerido: fraude, tratado como complemento y no como eje principal.\n"
)
diagnostico_lines.append(
    "- La predicción debe presentarse como componente exploratorio, no como objetivo central.\n\n"
)

diagnostico_lines.append("5) Salidas clave generadas\n")
diagnostico_lines.append("- fgj_top_delitos_clean.csv\n")
diagnostico_lines.append("- fgj_agregado_alcaldia_anio_clean.csv\n")
diagnostico_lines.append("- pobreza_agregado_alcaldia_anio_clean.csv\n")
diagnostico_lines.append("- panel_contemporaneo.csv\n")
diagnostico_lines.append("- panel_rezagado.csv\n")
diagnostico_lines.append("- top_asociaciones_robos_territoriales_contemporaneo.csv\n")
diagnostico_lines.append("- top_asociaciones_robos_territoriales_rezagado.csv\n")
diagnostico_lines.append("- resumen_metricas_proyecto.json\n")
diagnostico_lines.append("- diagnostico_final.txt\n")

diagnostico_final = "".join(diagnostico_lines)
save_txt(diagnostico_final, "diagnostico_final.txt")
print(diagnostico_final)

[OK] Guardado TXT: d:\DMProject\Documentation\eda_outputs\diagnostico_final.txt
DIAGNÓSTICO FINAL DEL NOTEBOOK

1) Calidad y cobertura de FGJ
- Filas válidas posteriores a limpieza: 2,086,791
- Años válidos de FGJ: [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
- Alcaldías detectadas en FGJ: 17
- Participación de robos territoriales sobre el total FGJ: 0.3041
- Participación de fraude sobre el total FGJ: 0.0734

2) Calidad y cobertura de pobreza
- Filas totales pobreza: 842,342
- Filas CDMX detectadas: 22,219
- Share CDMX sobre pobreza total: 0.0264
- Años detectados en pobreza CDMX: [2016, 2018, 2020]
- Alcaldías mapeadas en pobreza CDMX: 16

3) Temporalidad utilizable
- Años comunes contemporáneos: [2016, 2018, 2020]
- Años rezagados utilizables: [2021, 2022, 2023, 2024, 2025]
- Filas panel contemporáneo: 48
- Filas panel rezagado: 90

4) Recomendación metodológica
- Ventana principal recomendada: [2016, 2018, 2020], con análisis explicativo por alcaldía-año.
- Ventana 